# PSM Memory LOCOMO benchmark on Colab

This notebook installs the published npm packages, restores an already-ingested LOCOMO SQLite DB from Google Drive, and runs retrieval plus answer-generation benchmark evaluation.

Recommended Colab runtime: CPU is enough for benchmarking an existing DB. GPU is only needed if you re-run ingestion.

In [ ]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Keep caches under /content so the paths are predictable.
# Read the ingested DB from Drive and write benchmark outputs back to Drive.
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_DIR = "/content/drive/MyDrive/psm-memory-locomo"
DRIVE_DB = "/content/drive/MyDrive/Colab Notebooks/Data/locomo-psm-memory-ingest.db"
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p /content/psm-memory-work /content/locomo/results "$GDRIVE_DIR"
%cd /content/psm-memory-work
!npm init -y
!npm install @psm-memory/cli@latest @psm-memory/sdk@latest

In [ ]:
# Ensure the published CLI can prepare the embedding/model cache if benchmark helpers need it.
!npx psm-memory setup --pretty

In [ ]:
# Clone only for benchmark data and the Colab harness. The benchmark imports @psm-memory/sdk from npm.
!rm -rf /content/PSM
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs

In [ ]:
# Restore the already-ingested LOCOMO DB from Google Drive. This skips ingestion.
from pathlib import Path
import shutil

DRIVE_DB = globals().get("DRIVE_DB", "/content/drive/MyDrive/Colab Notebooks/Data/locomo-psm-memory-ingest.db")
DB = "/content/locomo/results/locomo-psm-memory-ingest.db"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"
GDRIVE_DIR = globals().get("GDRIVE_DIR", "/content/drive/MyDrive/psm-memory-locomo")

drive_db = Path(DRIVE_DB)
if not drive_db.exists():
    raise FileNotFoundError(f"Uploaded DB not found: {drive_db}")

Path(DB).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(drive_db, DB)
print(f"Restored benchmark DB from {drive_db}")
print(f"Local DB: {DB}")
print(f"DB size: {Path(DB).stat().st_size:,} bytes")

In [ ]:
# Evaluate evidence retrieval over the memory DB.
# Start with a cheap answer-evaluation smoke test before spending tokens on all 1,982 questions.
# Set ANSWER_LIMIT = 0 only after the smoke output looks good.
ANSWER_LIMIT = 50
ANSWER_TOP_K = 10
CHECKPOINT_EVERY = 5
RUN_LABEL = "full" if ANSWER_LIMIT == 0 else f"smoke-{ANSWER_LIMIT}"

OUT = f"/content/locomo/results/locomo-results-{RUN_LABEL}.json"
ANSWER_OUT = f"/content/locomo/results/locomo-answer-results-{RUN_LABEL}.json"
REPORT = f"/content/locomo/results/locomo-comparison-{RUN_LABEL}.md"
BASELINES = "/content/PSM/benchmark/locomo/baselines/memory-tools.json"
ANSWER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
JUDGE_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
GDRIVE_DIR = globals().get("GDRIVE_DIR", "/content/drive/MyDrive/psm-memory-locomo")

import getpass
import os
try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = os.environ.get('OPENROUTER_API_KEY') or (userdata.get('OPENROUTER_API_KEY') or '')
except Exception:
    pass
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OPENROUTER_API_KEY: ')

!set -e; \
node /content/psm-memory-work/locomo-benchmark.mjs evaluate --data "$DATA" --db "$DB" --out "$OUT" --top-k 3; \
node /content/psm-memory-work/locomo-benchmark.mjs answer-evaluate --data "$DATA" --db "$DB" --out "$ANSWER_OUT" --top-k $ANSWER_TOP_K --limit $ANSWER_LIMIT --answer-model "$ANSWER_MODEL" --judge-model "$JUDGE_MODEL" --checkpoint-every $CHECKPOINT_EVERY; \
node /content/psm-memory-work/locomo-benchmark.mjs report --psm "$ANSWER_OUT" --baselines "$BASELINES" --out "$REPORT"; \
cp "$OUT" "$GDRIVE_DIR/locomo-results-$RUN_LABEL.json"; \
cp "$ANSWER_OUT" "$GDRIVE_DIR/locomo-answer-results-$RUN_LABEL.json"; \
cp "$REPORT" "$GDRIVE_DIR/locomo-comparison-$RUN_LABEL.md"

In [ ]:
# Inspect output artifacts.
!ls -lh /content/locomo/results
!ls -lh "$GDRIVE_DIR"
import json
from pathlib import Path
run_label = globals().get('RUN_LABEL', 'smoke-50')
summary = Path('/content/locomo/results/ingest-summary.json')
results = Path(f'/content/locomo/results/locomo-results-{run_label}.json')
answer_results = Path(f'/content/locomo/results/locomo-answer-results-{run_label}.json')
report = Path(f'/content/locomo/results/locomo-comparison-{run_label}.md')
if summary.exists():
    print('ingest summary')
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])
if results.exists():
    print('retrieval evaluation summary')
    print(json.dumps(json.loads(results.read_text())['summary'], indent=2))
if answer_results.exists():
    print('answer evaluation summary')
    print(json.dumps(json.loads(answer_results.read_text())['summary'], indent=2))
if report.exists():
    print('comparison report')
    print(report.read_text()[:4000])

This notebook benchmarks the uploaded DB at `MyDrive/Colab Notebooks/Data/locomo-psm-memory-ingest.db`. It defaults to a 50-question answer-evaluation smoke test to control token spend. Review `locomo-answer-results-smoke-50.json`; set `ANSWER_LIMIT = 0` only when the smoke accuracy, context items, and judge reasoning look acceptable.